# 5. 完整训练流程

这是实际写模型最核心的部分。一个完整的训练流程包含以下步骤：

1. 准备数据（Dataset + DataLoader）
2. 定义模型（nn.Module）
3. 定义损失函数和优化器
4. 训练循环：前向传播 → 计算损失 → 反向传播 → 更新参数
5. 验证模型效果

这一章我们用一个简单的例子把整个流程串起来。

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

## 5.1 准备数据

这里我们用随机生成的数据来演示流程（后面第 6 章会学如何加载真实数据集）。

- 生成 1000 个样本，每个样本 20 个特征
- 二分类任务：标签为 0 或 1
- DataLoader 负责 batch 分批、打乱顺序

In [2]:
# 生成模拟数据
torch.manual_seed(42)  # 固定随机种子，保证结果可复现

# 1000个样本，每个20维特征
X = torch.randn(1000, 20)
# 标签：根据特征的简单规则生成
y = (X[:, 0] + X[:, 1] > 0).long()  # 前两个特征之和>0 → 标签1，否则0

# 划分训练集和测试集（80% / 20%）
split = 800
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# 用 TensorDataset 包装，再用 DataLoader 分批加载
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"训练集: {len(train_dataset)} 个样本")
print(f"测试集: {len(test_dataset)} 个样本")
print(f"每个 batch: 32 个样本")
print(f"训练 batch 数: {len(train_loader)}")

训练集: 800 个样本
测试集: 200 个样本
每个 batch: 32 个样本
训练 batch 数: 25


## 5.2 定义模型

一个简单的二分类网络：
- 输入 20 维
- 隐藏层 64 维 + ReLU
- 隐藏层 32 维 + ReLU
- 输出 2 维（二分类，两个类别的分数）

In [3]:
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(20, 64)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(32, 2)  # 二分类，输出2维

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        return x

model = Classifier()
print(model)
print(f"参数量: {sum(p.numel() for p in model.parameters()):,}")

Classifier(
  (fc1): Linear(in_features=20, out_features=64, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=32, out_features=2, bias=True)
)
参数量: 3,490


## 5.3 定义损失函数和优化器

- CrossEntropyLoss：分类任务的标准选择，内部自动做 Softmax
- Adam：自适应学习率优化器，lr=0.001 是常用默认值

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

## 5.4 训练循环

训练的核心流程，每个 epoch 做的事：

```
for epoch:
    model.train()            # 切换到训练模式
    for batch in dataloader:
        optimizer.zero_grad()  # 1. 梯度清零
        output = model(X)      # 2. 前向传播
        loss = criterion(...)  # 3. 计算损失
        loss.backward()        # 4. 反向传播（计算梯度）
        optimizer.step()       # 5. 更新参数
```

### model.train() 和 model.eval() 的区别

- train()：启用 Dropout 和 BatchNorm 的训练行为
- eval()：关闭 Dropout，BatchNorm 用全局统计量

### 为什么验证时用 torch.no_grad()

验证时不需要计算梯度，关闭后可以节省大量内存。

In [5]:
num_epochs = 20

for epoch in range(num_epochs):
    # === 训练阶段 ===
    model.train()  # 训练模式
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()          # 1. 梯度清零
        output = model(X_batch)        # 2. 前向传播
        loss = criterion(output, y_batch)  # 3. 计算损失
        loss.backward()                # 4. 反向传播
        optimizer.step()               # 5. 更新参数

        train_loss += loss.item()
        pred = output.argmax(dim=1)    # 取分数最高的类别作为预测
        train_correct += (pred == y_batch).sum().item()
        train_total += y_batch.size(0)

    # === 验证阶段 ===
    model.eval()  # 评估模式
    test_loss = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():  # 不计算梯度
        for X_batch, y_batch in test_loader:
            output = model(X_batch)
            loss = criterion(output, y_batch)

            test_loss += loss.item()
            pred = output.argmax(dim=1)
            test_correct += (pred == y_batch).sum().item()
            test_total += y_batch.size(0)

    # 打印每个 epoch 的结果
    train_acc = train_correct / train_total * 100
    test_acc = test_correct / test_total * 100
    if (epoch + 1) % 5 == 0 or epoch == 0:  # 每5个epoch打印一次，加上第1个
        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"训练损失: {train_loss/len(train_loader):.4f} 训练准确率: {train_acc:.1f}% | "
              f"测试损失: {test_loss/len(test_loader):.4f} 测试准确率: {test_acc:.1f}%")

Epoch [1/20] 训练损失: 0.6540 训练准确率: 67.0% | 测试损失: 0.6108 测试准确率: 81.0%
Epoch [5/20] 训练损失: 0.1728 训练准确率: 96.4% | 测试损失: 0.1726 测试准确率: 94.0%
Epoch [10/20] 训练损失: 0.0506 训练准确率: 99.6% | 测试损失: 0.1143 测试准确率: 95.5%
Epoch [15/20] 训练损失: 0.0200 训练准确率: 100.0% | 测试损失: 0.1054 测试准确率: 96.5%
Epoch [20/20] 训练损失: 0.0094 训练准确率: 100.0% | 测试损失: 0.1085 测试准确率: 97.0%


## 5.5 GPU 训练

上面的例子是在 CPU 上跑的。实际训练时，只需要做 3 处修改就能用 GPU：

1. 模型移到 GPU：model.to('cuda')
2. 数据移到 GPU：X_batch.to('cuda'), y_batch.to('cuda')

其他代码完全不用改，PyTorch 自动处理。

In [7]:
# GPU 训练示例（只展示关键改动）

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"使用设备: {device}")

model_gpu = Classifier().to(device)  # 模型移到 GPU
optimizer = torch.optim.Adam(model_gpu.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# 训练循环中只需改数据转移这一行
# for X_batch, y_batch in train_loader:
#     X_batch, y_batch = X_batch.to(device), y_batch.to(device)  # 数据移到 GPU
#     ... 其余完全一样

print("GPU 训练只需加 .to(device)，其余代码不变")

使用设备: cuda
GPU 训练只需加 .to(device)，其余代码不变
